In [ ]:
# ── Harness setup ─────────────────────────────────────────────────────────────
import sys, os, json, pathlib
sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, model_fast
env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())
from elasticsearch import Elasticsearch
es = Elasticsearch(os.environ['ES_URL'], api_key=os.environ['ES_API_KEY'], request_timeout=60)
client = llm_client()
FAST = model_fast()
TRACES = pathlib.Path('/home/elastic/.traces'); TRACES.mkdir(exist_ok=True)

DEV_QUERIES = [
    {'query_id': 'fail-01', 'query_text': 'money moved around to hide where it came from'},
    {'query_id': 'fail-02', 'query_text': 'when funds flow through several companies before reaching a final destination'},
    {'query_id': 'fail-03', 'query_text': 'transaction timing designed to look like normal business activity'},
    {'query_id': 'fail-04', 'query_text': 'how a bank employee gets tricked into redirecting a payment'},
]
print('Harness ready. Dev queries loaded.')

In [ ]:
# ── Baseline: run raw queries ─────────────────────────────────────────────────
for q in DEV_QUERIES:
    r = es.search(index='cortex-corpus-live', body={
        'retriever': {'standard': {'query': {'semantic': {'field': 'body', 'query': q['query_text']}}}},
        'size': 5, '_source': ['doc_id']
    })
    top5 = [h['_source'].get('doc_id','') for h in r['hits']['hits']]
    print(f"{q['query_id']}: top-5 = {top5}")

In [ ]:
# ── YOUR WORK ── Implement rewrite() ─────────────────────────────────────────
# Constraint: the index is not touched. Only the query changes.
#
# Options:
#   HyDE: generate a hypothetical excerpt, search on that
#   Query expansion: add domain terms
#   Multi-query: generate several variants, merge

def rewrite(query_text: str) -> str:
    """Return a rewritten query string that should improve recall."""
    # ← YOUR IMPLEMENTATION
    resp = client.chat.completions.create(
        model=FAST,
        messages=[{
            'role': 'user',
            'content': (
                'Write a one-paragraph excerpt from a Cortex Bank compliance document '
                'that directly answers this question:\n' + query_text + '\nExcerpt only.'
            )
        }],
        temperature=0,
    )
    return resp.choices[0].message.content or query_text

print('rewrite() defined.')

In [ ]:
# ── Run rewrite and save trace ────────────────────────────────────────────────
trace_path = TRACES / 'recall-rewrite.jsonl'
with open(trace_path, 'w') as f:
    for q in DEV_QUERIES:
        rewritten = rewrite(q['query_text'])
        entry = {'query_id': q['query_id'], 'query_text': q['query_text'], 'rewritten_query': rewritten[:500]}
        f.write(json.dumps(entry) + '\n')
        r = es.search(index='cortex-corpus-live', body={
            'retriever': {'standard': {'query': {'semantic': {'field': 'body', 'query': rewritten}}}},
            'size': 5, '_source': ['doc_id']
        })
        top5 = [h['_source'].get('doc_id','') for h in r['hits']['hits']]
        print(f"{q['query_id']}: {top5}")
print(f'Trace saved to {trace_path}. Select Check.')